# 1. Import the necessary libraries

In [ ]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from problems.benchmark_problems import get_problem
from problems.microgrid_function import microgrid_function
from runners import run_mesh, run_nsga2
from tuners import fine_tune_mesh, fine_tune_nsga2

from contextlib import redirect_stdout
from io import StringIO
from joblib import Parallel, delayed
from optuna.pruners import NopPruner
from pymoo.indicators.hv import HV

import numpy as np

# 2. Define the general configuration of the experiments

In [ ]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 2

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [('dtlz1', 2, 10, None), ('dtlz2', 2, 10, None)]

# Execution configuration
num_runs = 2 # Number of runs
max_fitness_eval = 10000 # Maximum fitness evaluations (not used if it is None)
population_size = 100 # Population size
random_state = None # Set a seed for random numbers (not used if it is None)

results_folder = '../results' # Folder to store the results
tuning_folder = '../hyperparams' # Folder to get the tuned parameters

# Fine tuning configuration
n_trials = 30 # Number of trials for Optuna
n_steps = 5 # Number of rounds in a trial
optuna_pruner = NopPruner() # Optuna's pruner to prune bad trials according to a metric
# optuna_pruner = optuna.pruners.WilcoxonPruner(p_threshold=0.05, n_startup_steps=n_steps//2)
hypercube_vertex = 1000 # Consider the vertex of the hypercube
######################################################

ref_point = [np.array([hypercube_vertex] * exp[1]) for exp in experiments] # Reference point for hypervolume calculation
indicators = [HV(ref_point=rf) for rf in ref_point] # Define the indicator to calculate the volume

# 3 Define auxiliar functions

In [ ]:
# Function to get the fitness function configuration
def get_exp_problem(func_name, objective_dim, position_dim, wfg_k):
	select_bat = {'Lead_Acid': 0, 'Li-ion': 1, 'ZEBRA': 2, 'NaS': 3, 'NiCd': 4, 'NiMH': 5, 'RFV': 6, 'ZnBr': 7}
	# Microgrid problem
	if func_name in select_bat:
		load = np.genfromtxt('../seasonal_data/load.txt')
		temperature = np.genfromtxt('../seasonal_data/temperature.txt')
		solar_data = np.genfromtxt('../seasonal_data/irradiance.txt')
		wind_data = np.genfromtxt('../seasonal_data/wind.txt')
		def microgrid_func(args):
			return microgrid_function(args[0], args[1], args[2], select_bat[func_name], load, temperature, solar_data, wind_data)
		lower_bound_array = np.array([1.0, 1.0, 1.0])
		upper_bound_array = np.array([2000.0, 2000.0, 2000.0])
		return {'fitness': microgrid_func,
		        'objective_dim': objective_dim,
		        'position_dim': position_dim,
		        'lower_bound_array': lower_bound_array,
		        'upper_bound_array': upper_bound_array}
	# Benchmark problems
	else:
		func, lower_bound_array, upper_bound_array = get_problem(func_name, n_var=position_dim, n_obj=objective_dim, wfg_k=wfg_k)
		return {'fitness': func,
		        'objective_dim': objective_dim,
		        'position_dim': position_dim,
		        'lower_bound_array': lower_bound_array,
		        'upper_bound_array': upper_bound_array}

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		f = StringIO()
		with redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 4. Run experiments with the algorithms (in parallel)

## 4.1 MESH

In [ ]:
#################### CUSTOMIZABLE ####################
# MESH fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory
mesh_global_best_type = [0] # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_dm_pool_type = [0] # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_evolution_type = [0] # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)


# MESH tunable parameters
# OBS: The function "run_mesh" and "run_mesh_old" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
mesh_communication_probability = 0.8 # Communication probability
mesh_mutation_rate = 0.9 # Mutation rate
mesh_personal_guide_array_size = 1 # Number of personal guides
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
mesh_fine_tuning_params_list = [[
    {
        'name': f'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
    get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
    {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
    {
        'memory_size': mesh_memory_size,
        'global_best_attribution_type': gb_type,
        'differential_mutation_pool_type': pool_type,
        'differential_mutation_type': de_type
    },
    indicators[i]
    ]
     for i, exp in enumerate(experiments)
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
if __name__ == "__main__":
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_mesh, params)
        for params in mesh_fine_tuning_params_list
    )
return_values

### Run

In [ ]:
# Set the list of parameters
mesh_run_params_list = [[
    {'name': f'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}',
     'results_folder': results_folder,
     'fine_tuning_folder': tuning_folder,
     'num_runs': num_runs,
     'max_fitness_eval': max_fitness_eval,
     'population_size': population_size,
     'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {'memory_size': mesh_memory_size,
      'global_best_attribution_type': gb_type,
      'differential_mutation_pool_type': pool_type,
      'differential_mutation_type': de_type,
      'communication_probability': mesh_communication_probability,
      'mutation_rate': mesh_mutation_rate,
      'personal_guide_array_size': mesh_personal_guide_array_size}
    ]
     for exp in experiments
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_mesh, params)
        for params in mesh_run_params_list
    )
return_values

## 4.2 NSGA-II

In [ ]:
#################### CUSTOMIZABLE ####################
# NSGA2 fixed parameters


# NSGA2 tunable parameters
nsga2_recombination_probability = 0.45
nsga2_eta_recombination = 19
nsga2_mutation_probability = 0.06
nsga2_eta_mutation = 14.7
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
nsga2_fine_tuning_params_list = [[
    {
        'name': f'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
     {},
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_nsga2, params)
        for params in nsga2_fine_tuning_params_list
    )
    print(return_values)

### Run

In [ ]:
# Set the list of parameters
nsga2_run_params_list = [[
    {
        'name': f'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}',
        'results_folder': results_folder,
        'fine_tuning_folder': tuning_folder,
        'num_runs': num_runs,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {
        'recombination_probability': nsga2_recombination_probability,
        'eta_recombination': nsga2_eta_recombination,
        'mutation_probability': nsga2_mutation_probability,
        'eta_mutation': nsga2_eta_mutation}
    ]
     for exp in experiments
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_nsga2, params)
        for params in nsga2_run_params_list
    )
    print(return_values)

## 4.3 MOPSO